<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div><strong>Universidad Externado de Colombia</strong><br>
  <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
  <span>Docente: Wilmer Pineda-Ríos</span></div>
  <img src="../../assets/brand/logo-externado.png" width="190">
</div>

# Sesión 1 — Árboles de regresión

**Objetivo.** Explicar cómo CART construye promedios locales y evaluar si esa flexibilidad mejora una referencia honesta.

## 1. De clasificación a regresión

En una hoja de clasificación elegíamos una clase o proporción. En regresión predecimos la media de los valores que llegan a la hoja. La división busca minimizar la suma de errores cuadrados dentro de las regiones.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Un árbol de regresión a mano

Construimos un caso unidimensional para observar la función escalonada.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
x_demo = np.linspace(0, 10, 80)
y_demo = 10 + 4 * (x_demo > 3) + 7 * (x_demo > 7) + rng.normal(0, 1.2, len(x_demo))

demo = pd.DataFrame({"x": x_demo, "y": y_demo})
demo.head()

In [ ]:
def split_sse(data, threshold):
    left = data[data["x"] <= threshold]["y"]
    right = data[data["x"] > threshold]["y"]
    if len(left) == 0 or len(right) == 0:
        return np.inf
    return ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()

candidates_demo = demo["x"].iloc[1:-1]
scores = pd.DataFrame({"corte": candidates_demo, "SSE": [split_sse(demo, t) for t in candidates_demo]})
best = scores.loc[scores["SSE"].idxmin()]
best

In [ ]:
tree_demo = DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE).fit(demo[["x"]], demo["y"])
pred_demo = tree_demo.predict(demo[["x"]])
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_demo, y_demo, s=18, alpha=.6, label="observaciones")
ax.step(x_demo, pred_demo, color="#0f766e", linewidth=2.5, where="mid", label="árbol")
ax.set(xlabel="x", ylabel="y", title="El árbol aprende una función constante por regiones")
ax.legend()
plt.show()

**Pausa conceptual.** ¿Por qué la media de cada hoja minimiza la SSE? ¿Qué cambiaría si la función objetivo fuera el error absoluto?

## 3. Caso: demanda diaria de bicicletas

La decisión es anticipar cuántas bicicletas requiere la operación. El archivo corresponde al conjunto Bike Sharing de UCI (DOI: 10.24432/C5W894).

In [ ]:
candidates = [
    Path("../../datasets/public/Bike_Sharing_Day.csv"),
    Path("datasets/public/Bike_Sharing_Day.csv"),
    Path("../datasets/public/Bike_Sharing_Day.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("No se encontró Bike_Sharing_Day.csv")

df = pd.read_csv(data_path, parse_dates=["dteday"]).sort_values("dteday").reset_index(drop=True)
df.head()

In [ ]:
df[["casual", "registered", "cnt"]].assign(
    suma=lambda d: d["casual"] + d["registered"],
    coincide=lambda d: d["casual"] + d["registered"] == d["cnt"],
).head()

`casual` y `registered` no son predictores legítimos: su suma es exactamente el objetivo. Usarlos produciría fuga.

In [ ]:
target = "cnt"
leakage = ["casual", "registered"]
drop_columns = ["instant", "dteday", target, *leakage]
X = df.drop(columns=drop_columns)
y = df[target]

categorical = ["season", "mnth", "weekday", "weathersit"]
numeric = [c for c in X.columns if c not in categorical]

cut = int(len(df) * 0.80)
X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

preprocess = ColumnTransformer(
    [("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    remainder="passthrough",
)
cv = TimeSeriesSplit(n_splits=5)
X_train.shape, X_test.shape, (df.loc[cut, "dteday"], df["dteday"].max())

In [ ]:
def metrics(name, y_true, y_pred):
    return {
        "modelo": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

## 4. Baselines y primer árbol

In [ ]:
models = {
    "mediana": DummyRegressor(strategy="median"),
    "lineal": Pipeline([("prep", preprocess), ("model", LinearRegression())]),
    "arbol_d3": Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(max_depth=3, min_samples_leaf=15, random_state=RANDOM_STATE))]),
}
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append(metrics(name, y_test, pred))
pd.DataFrame(results).set_index("modelo").round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
dates = df.loc[cut:, "dteday"]
ax.plot(dates, y_test, label="real", color="#172029")
ax.plot(dates, predictions["arbol_d3"], label="árbol", color="#0f766e")
ax.set(title="Demanda real y estimada en el periodo de prueba", ylabel="bicicletas")
ax.legend()
plt.show()

## 5. Lectura técnica y de negocio

- MAE se expresa en bicicletas y será la métrica principal.
- RMSE aumenta el peso de errores grandes.
- Un árbol no extrapola fuera de los valores aprendidos.
- La evaluación cronológica representa mejor una decisión futura que mezclar fechas al azar.

## 6. Salida

1. ¿Qué predice exactamente una hoja?
2. ¿Qué optimiza un corte?
3. ¿Por qué `casual` y `registered` son fuga?
4. ¿Qué gana y qué pierde el árbol frente al modelo lineal?